## Libraries

In [1]:
import os

In [2]:
run = 'local' #local or cloud
os.environ["APP_ENV"] = run

In [3]:
import sys
sys.path.append('..')

In [4]:
from src.core_grass import*
%matplotlib inline
#%load_ext autoreload
#%autoreload 2

In [5]:
import subprocess
result = subprocess.run(["grass", "--config", "python_path"],
                        check=True, text=True, capture_output=True)
sys.path.append(result.stdout.strip())

# import GRASS Python packages
from collections import defaultdict
import grass.script as gs
import grass.jupyter as gj

# Parameters

In [6]:
#set project variables
Project_Area = 'essex'
projection = '3424' #set to None if you want to use the DEM's original projection
res = '1m'

## Set variables analysis
# source = 'wbd' #wbd or user-defined
# data_scale = 'HUC12' #other options HUC12, HUC10, HUC8 OR the field name of the source data to be split <-- case sensitive
# select_data = '030300020407' #     # Value within the filter_field used for data selection 
# to_headwaters = False #False if only interested in local huc area
# input_shp_name = None #None or must be populated populated with a shapefile polygons if data_scale != HUCXX
# buffer = 2000
# auto_delete = False

In [7]:
## Paths:
parent_dir = pl.Path(os.getcwd()).parent/'data'
grass_data_dir = pl.Path(os.getcwd()).parent/'grassdata'
current_dir = pl.Path(os.getcwd())
outputs_dir =parent_dir/'Rasters'/Project_Area/'Outputs'
vector_dir = parent_dir/'Vectors'/Project_Area
raster_dir = parent_dir/'Rasters'/Project_Area
huc_path = vector_dir/'{}_aoi.geojson'.format(Project_Area)
git_data_repo = current_dir/'data_repository'
# if input_shp_name:
#     input_shp = vector_dir/input_shp_name
directory = outputs_dir

### Start new grass session

In [8]:
# define GRASS data settings (adapt to your needs)
location = Project_Area

In [9]:
# Create a project if it doesn't exist
if os.path.exists(grass_data_dir/location):
    pass
else:
    if len(projection) > 4:
        wkt_text = pyproj.CRS.from_authority("esri", int(projection)).to_wkt()
        with open(vector_dir/'{}_prj.wkt'.format(Project_Area), 'w') as t:
            t.write(wkt_text)
        gs.create_project(path=grass_data_dir,
                      name=location,
                      wkt=vector_dir/'{}_prj.wkt'.format(Project_Area))
    else:
        gs.create_project(path=grass_data_dir,
                  name=location,
                  epsg=projection)

In [10]:
# Start GRASS in default project mapset
session = gj.init(grass_data_dir/location)

#### Review existing files

In [11]:
#list all grass files
grass_maps = list_existing_grass(print_it=True)

Available vector maps:
BloomfieldAve_AOI@PERMANENT
BloomfieldAve_AOI_ces@PERMANENT
BloomfieldAve_AOI_cesd@PERMANENT
BloomfieldAve_AOI_cesdb@PERMANENT
BloomfieldAve_AOI_cesdbb@PERMANENT
BloomfieldAve_AOI_cesdbbf@PERMANENT
BloomfieldAve_AOI_cesdbbfe@PERMANENT
BloomfieldAve_AOI_cesdbbfec@PERMANENT
BloomfieldAve_AOI_cesdbbfec8f@PERMANENT
BloomfieldAve_AOI_cesdbbfec8f8r@PERMANENT
BloomfieldAve_AOI_cln@PERMANENT
BloomfieldAve_AOI_exp@PERMANENT
BloomfieldAve_AOI_raw@PERMANENT
EC_EnvJustice_combined_streesor_summary@PERMANENT
GI_pnts@PERMANENT
GI_pnts_raw@PERMANENT
GI_pnts_raw_back@PERMANENT
ParkAve_AOI@PERMANENT
ParkAve_AOI_ces@PERMANENT
ParkAve_AOI_cesd@PERMANENT
ParkAve_AOI_cesdb@PERMANENT
ParkAve_AOI_cesdbb@PERMANENT
ParkAve_AOI_cesdbbf@PERMANENT
ParkAve_AOI_cesdbbfe@PERMANENT
ParkAve_AOI_cesdbbfec@PERMANENT
ParkAve_AOI_cesdbbfec8f@PERMANENT
ParkAve_AOI_cesdbbfec8f8r@PERMANENT
ParkAve_AOI_cesdf@PERMANENT
ParkAve_AOI_cesdfb@PERMANENT
ParkAve_AOI_cesdfe@PERMANENT
ParkAve_AOI_cesdfeb@PERMANEN

In [12]:
reg = reg = gs.parse_command('g.region',flags='gp')
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

In [13]:
threshold = 5000//(cell_size**2) #5000 sf
main_select_data= 'essex_nj'
aoi = 'BloomfieldAve_AOI' #ParkAve_AOI, SOrange_AOI BloomfieldAve_AOI
select_data = main_select_data+'_'+aoi
gi_raw = f'{aoi}_raw'

In [14]:
#standard variables
dem = 'dem_essex_nj'
c_lines = f'v_carve_{select_data}'
accum = 'row_accum'
drain_dir = 'row_drain_dir'
r_basins = f'r_basins_{select_data}'
v_basins = f'v_basins_{select_data}'
sub_basins = f'subbasins_{select_data}'
v_streams = f'stream_{select_data}'
r_streams = f'r_streams_{select_data}'
r_streams_o = f'r_streams_o_{select_data}'
twi = f'r_twi_{select_data}'
v_outlet = f'outlet_{select_data}'
v_inlet = f'inlet_{select_data}'

In [15]:
dem

'dem_essex_nj'

## Prep Work

In [16]:
gs.run_command('r.mask',flags='r')

In [17]:
gs.run_command('g.region',raster = dem,align=dem,zoom=dem)

In [18]:
gs.run_command("v.clip", input="ROWLimits_nonParcel", clip=aoi, output=select_data)

In [19]:
gs.run_command("v.clip", input="roads", clip=aoi, output=f"roads_{select_data}")

In [20]:
gs.run_command('r.mask',vector = select_data)
gs.run_command('g.region',raster = dem,align=dem,zoom=dem)
reg = gs.parse_command('g.region',raster = dem,align=dem,zoom=dem,flags='gp')

In [21]:
gs.run_command('r.watershed',elevation = dem, threshold=threshold,accumulation = accum, drainage=drain_dir, stream = r_streams,flags='a')

In [22]:
import re
import subprocess

# 1. Start the stream extraction process asynchronously
p = gs.start_command('r.stream.extract',elevation=dem,accumulation=accum,threshold=threshold,stream_vector =v_streams, direction=drain_dir, memory = 3000,stderr=subprocess.PIPE)

# 2. Read the output in real-time
while True:
    line = p.stderr.readline()
    if not line and p.poll() is not None:
        break
    if line:
        # Decode bytes to string and print (or parse for percentages)
        print(line)

# 3. Finalize process check
rc = p.returncode
if rc == 0:
    print("Process completed successfully.")
else:
    print(f"Process failed with return code {rc}.")

Loading input raster maps...



Initializing A* search...



A* Search...



Extracting streams...



Thinning stream segments...



Writing output raster maps...



Writing vector map <stream_essex_nj_BloomfieldAve_AOI>...


         will be overwritten



Writing attribute data...

Building topology for vector map

<stream_essex_nj_BloomfieldAve_AOI@PERMANENT>...

Registering primitives...



Process completed successfully.


In [23]:
gs.run_command("v.db.renamecolumn", map=v_streams, column="stream_type,strm_type")

In [24]:
gs.run_command("v.clean", input=v_streams, output=v_streams+'_cln', tool="rmline")
gs.run_command('v.out.ogr',input=v_streams+'_cln',output=vector_dir/f'flow_{v_streams}.shp',type= 'line',format='ESRI_Shapefile')
#gs.run_command('v.out.ogr',input=v_streams,output=vector_dir/f'flow_{v_streams}.GeoJSON',type= 'line',format='GeoJSON')

In [25]:
gs.run_command("v.extract", input=v_streams, output=v_inlet+'_line',type = 'line',where = "(strm_type = 'start')") 
gs.run_command("v.to.points",input=v_inlet+'_line', output=v_inlet, use='start')

In [ ]:
gs.run_command('r.stream.basins',direction =drain_dir,points =v_inlet,basins=r_basins)
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [ ]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'TDA_less_than_5000_{aoi}.shp',type= 'area',format='ESRI_Shapefile')

In [ ]:
aoi

### SLOPE

In [ ]:
gs.run_command('r.slope.direction',elevation = dem,direction =drain_dir,dir_type ='auto',slope_measure  ='percent',steps =7, output ='slope_dir',flags='a')

In [ ]:
gs.mapcalc("steep_10_pct = if({0} > 0.15, 1, null())".format('slope_dir'))

In [ ]:
gs.run_command('r.to.vect',input = 'steep_10_pct',output = f'steep_10_pct_{aoi}', type='area',column='steep_col', flags='s')

In [ ]:
gs.run_command('v.out.ogr',input=f'steep_10_pct_{aoi}',output=vector_dir/f'slope_more_than_10_pct_{aoi}.shp',type= 'area',format='ESRI_Shapefile')

## Work for GI Screening

In [ ]:
#clean
gs.run_command("v.multi2singlepart", input=gi_raw, output=gi_raw.replace('raw','exp'))
gs.run_command("v.clean", input=gi_raw.replace('raw','exp'),output=gi_raw.replace('raw','cln'), tool='rmdangle,rmbridge,bpol,rmarea',threshold='10,20,0,5',flags='c')

In [ ]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cln'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cln')}.shp',type= 'area',format='ESRI_Shapefile')

In [ ]:
#remove steep areas
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','cln'), binput=f'steep_10_pct_{aoi}', operator="not", output=gi_raw.replace('raw','ces'),flags='t')

In [ ]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','ces'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','ces')}.shp',type= 'area',format='ESRI_Shapefile')

In [ ]:
#remove less than 5000 sf areas
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','ces'), binput=v_basins, operator="not", output=gi_raw.replace('raw','cesd'),flags='t')

In [ ]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesd'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesd')}.shp',type= 'area',format='ESRI_Shapefile')

In [ ]:
#remove false connections
gs.run_command('v.buffer',input=gi_raw.replace('raw','cesd'), output=gi_raw.replace('raw','cesdb'),distance = -2)
gs.run_command('v.buffer',input=gi_raw.replace('raw','cesdb'), output=gi_raw.replace('raw','cesdbb'),distance = 2)

In [ ]:
#separate by flow path
gs.run_command('v.buffer',input=v_streams+'_cln',output=v_streams+'_cln'+'_buffer',distance = 0.1)
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','cesdbb'), binput=v_streams+'_cln'+'_buffer', operator="not", output=gi_raw.replace('raw','cesdbbf'),flags='t')

In [ ]:
#clean
gs.run_command("v.multi2singlepart", input=gi_raw.replace('raw','cesdbbf'), output=gi_raw.replace('raw','cesdbbfe'))
gs.run_command("v.clean", input=gi_raw.replace('raw','cesdbbfe'),output=gi_raw.replace('raw','cesdbbfec'), tool='rmdangle,rmbridge,bpol,rmarea',threshold='10,20,0,5',flags='c')

In [ ]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesdbbfec'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec')}.shp',type= 'area',format='ESRI_Shapefile')

In [ ]:
#select down
gs.run_command('v.buffer',input=v_streams+'_cln',output=v_streams+'_cln'+'select_buffer',distance = 8)
gs.run_command('v.buffer',input=f"roads_{select_data}",output=f"roads_{select_data}"+'select_buffer',distance = 8)

In [ ]:
gs.run_command("v.select", ainput=gi_raw.replace('raw','cesdbbfec'), binput=v_streams+'_cln'+'select_buffer', output=gi_raw.replace('raw','cesdbbfec8f'), operator="intersects",flags='t')
gs.run_command("v.select", ainput=gi_raw.replace('raw','cesdbbfec8f'), binput=f"roads_{select_data}"+'select_buffer', output=gi_raw.replace('raw','cesdbbfec8f8r'), operator="intersects",flags='tr')

In [ ]:
gs.run_command('v.db.addtable',map=gi_raw.replace('raw','cesdbbfec8f8r'))

In [ ]:
#get raster info
gs.run_command("r.resample", input="TCC", output="TCC_sm")
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdbbfec8f8r'), raster="TCC_sm", column_prefix="TCC_Pct",method ='average',flags='cd')

In [ ]:
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdbbfec8f8r'), raster="EC_CST", column_prefix="EC_CST_Rank",method ='average',flags='cd')

In [ ]:
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdbbfec8f8r'), raster="Wtr_table_depth_ft", column_prefix="Dp2Wtr_ft",method ='minimum',flags='cd')

In [ ]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesdbbfec8f8r'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson').to_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.shp')

## WORK FOR GI AREAS Attribution

In [ ]:
# Load or create your GeoDataFrame
gdf_cent = gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.geojson')

# Create a new column with centroid points
gdf_cent["geometry"] = gdf_cent.geometry.representative_point()
gdf_cent["basin_col"] = gdf_cent.index+1

gdf_cent.to_file(vector_dir/'GI_Opps_pnts.shp')

In [ ]:
gs.run_command('v.import', input= vector_dir/'GI_Opps_pnts.shp', output= 'GI_pnts_raw')

In [ ]:
gs.run_command('g.copy', vector='GI_pnts_raw,GI_pnts_raw_back')

In [ ]:
gs.run_command('r.stream.snap',input = 'GI_pnts_raw', output='GI_pnts',stream_rast = r_streams,radius = 50/cell_size)

In [ ]:
#gs.run_command('v.db.connect', map='GI_pnts',table='GI_pnts_raw',database ='$GISDBASE/$LOCATION_NAME/PERMANENT/sqlite/sqlite.db', overwrite =True)
gs.run_command('v.db.addtable', map='GI_pnts')
gs.run_command('v.db.join', map='GI_pnts', column='cat', other_table='GI_pnts_raw_back', other_column='cat')

In [ ]:
gs.run_command('r.stream.basins',points = 'GI_pnts', direction=drain_dir,basins=r_basins+'GI')

In [ ]:
gs.run_command('r.to.vect',input = r_basins+'GI',output = v_basins+'GI', type='area',column='basin_col', flags='s')

In [ ]:
gs.run_command('v.out.ogr',input=v_basins+'GI',output=vector_dir/f'{v_basins}_GI.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}_GI.geojson').to_file(vector_dir/f'{v_basins}_GI.shp')

In [ ]:
basin_data = gpd.read_file(vector_dir/f'{v_basins}_GI.shp')
basin_data.set_index('basin_col',inplace=True)
#attach shape info
basin_data['TDA_sf'] = basin_data.geometry.area
basin_data['TDA_acre'] = basin_data.geometry.area/43560
tda_ref = basin_data[['TDA_sf']].to_dict()

In [ ]:
GI_data = gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdbbfec8f8r')}.shp')
GI_data['basin_col'] = GI_data.index+1
#attach shape info
GI_data['Area_sf'] = GI_data.geometry.area
GI_data['Perim_ft'] = GI_data.geometry.length
GI_data['Width_ft'] = (2*GI_data.geometry.area)/GI_data.geometry.length
GI_data['Length_ft'] = GI_data.geometry.area/GI_data['Width_ft']
GI_data['LR'] = GI_data.apply(lambda x: (tda_ref['TDA_sf'][x.basin_col]/x.Area_sf) if x.basin_col in tda_ref['TDA_sf'].keys() else 0,axis=1)
GI_data['TDA_sf'] = GI_data.apply(lambda x: (tda_ref['TDA_sf'][x.basin_col]) if x.basin_col in tda_ref['TDA_sf'].keys() else 0,axis=1)
GI_data['TDA_acres'] = GI_data.apply(lambda x: (tda_ref['TDA_sf'][x.basin_col]/43560) if x.basin_col in tda_ref['TDA_sf'].keys() else 0,axis=1)

In [ ]:
#lat long
GI_data['Lat'] = GI_data.geometry.centroid.to_crs(4269).y
GI_data['Lon'] = GI_data.geometry.centroid.to_crs(4269).x

In [ ]:
#filter out widths less than 3.5 feet.
GI_data_cut = GI_data.loc[GI_data['Width_ft']>=3.5]

In [ ]:
#add soil data, soil contamination (with 50' buffer), floodplain data and blank columns
soils = gpd.read_file(vector_dir/'soil_out.shp').to_crs(GI_data_cut.crs)[['HYDRO_GRP', 'geometry']]
env_soils = gpd.read_file(vector_dir/'EC_Groundwater_Contamination_Areas_(CEA).shp').to_crs(GI_data_cut.crs)[['CEA_NAME', 'geometry']]
env_soils_dp_dct = gpd.read_file(vector_dir/'EC_Groundwater_Contamination_Areas_(CEA).shp').to_crs(GI_data_cut.crs)[['CEA_NAME', 'DEPTH']].set_index('CEA_NAME').to_dict()
floodplain = gpd.read_file(vector_dir/'EFFEC_PRELIM_COMB_S_FLD_HAZ_AR.shp').to_crs(GI_data_cut.crs)[['FLD_ZONE','ZONE_SUBTY', 'geometry']]
bus_stops = gpd.read_file(vector_dir/'BUS_STOPS_clip.shp').to_crs(GI_data_cut.crs)[['STOP_NUM','geometry']]
zoning = gpd.read_file(vector_dir/'EssexZoning_Dissolved.shp').to_crs(GI_data_cut.crs)[['NV5_ZONING','geometry']]
truck_traffic = gpd.read_file(vector_dir/'EC_EnvJustice_combined_streesor_summary.shp').to_crs(GI_data_cut.crs)[['TRUCKTRA_4','geometry']]

In [ ]:
def add_dpt(x,dpt_dict):
    if pd.isna(x):
        dpt = np.nan
    elif x not in dpt_dict.keys():
        dpt_ls = []
        a = x.split(', ')
        for e in a:
            dpt_ls.append(str(dpt_dict[e]))
        dpt = ", ".join(dpt_ls)
    else:
        dpt = dpt_dict[x]
    return dpt



In [ ]:
#join attributes
#soils
GI_data_cut_s = gpd.sjoin(GI_data_cut,soils,how="inner", predicate="intersects")
aggregated_attributes = (GI_data_cut_s.groupby(GI_data_cut_s.index)["HYDRO_GRP"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_s = GI_data_cut.merge(aggregated_attributes, left_index=True, right_index=True, how="left")

#env soils
GI_data_cut_se = gpd.sjoin(GI_data_cut_s,env_soils,how="inner", predicate="intersects",rsuffix='ei')
aggregated_attributes = (GI_data_cut_se.groupby(GI_data_cut_se.index)["CEA_NAME"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_se = GI_data_cut_s.merge(aggregated_attributes, left_index=True, right_index=True, how="left")
GI_data_cut_se['In_CEA'] = GI_data_cut_se["CEA_NAME"].apply(lambda x: "No" if pd.isna(x) else "Yes")
GI_data_cut_se = GI_data_cut_se.drop(columns="CEA_NAME")

#50'
GI_data_cut_see = gpd.sjoin(GI_data_cut_se,env_soils,how="inner", predicate="dwithin",rsuffix='e',distance=50)
aggregated_attributes = (GI_data_cut_see.groupby(GI_data_cut_see.index)["CEA_NAME"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_see = GI_data_cut_se.merge(aggregated_attributes, left_index=True, right_index=True, how="left")
GI_data_cut_see['Withn50CEA'] = GI_data_cut_see["CEA_NAME"].apply(lambda x: "No" if pd.isna(x) else "Yes")
GI_data_cut_see['CEA_Res_Dp'] = GI_data_cut_see["CEA_NAME"].apply(lambda x: add_dpt(x,env_soils_dp_dct['DEPTH']))

#floodplain
GI_data_cut_seef = gpd.sjoin(GI_data_cut_see,floodplain,how="inner", predicate="intersects",rsuffix='f')
GI_data_cut_seef['FLD_HAZ'] = GI_data_cut_seef.apply(lambda x: x.FLD_ZONE + ' - ' + x.ZONE_SUBTY if not pd.isna(x.ZONE_SUBTY) else x.FLD_ZONE, axis=1)
aggregated_attributes = (GI_data_cut_seef.groupby(GI_data_cut_seef.index)["FLD_HAZ"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_seef = GI_data_cut_see.merge(aggregated_attributes, left_index=True, right_index=True, how="left")

In [ ]:
#bus stop
GI_data_cut_seefb = gpd.sjoin(GI_data_cut_seef,bus_stops,how="inner", predicate="dwithin",rsuffix='b',distance=50)
aggregated_attributes = (GI_data_cut_seefb.groupby(GI_data_cut_seefb.index)["STOP_NUM"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_seefb = GI_data_cut_seef.merge(aggregated_attributes, left_index=True, right_index=True, how="left")
GI_data_cut_seefb['Bus_20ft'] = GI_data_cut_seefb["STOP_NUM"].apply(lambda x: "No" if pd.isna(x) else "Yes")
GI_data_cut_seefb = GI_data_cut_seefb.drop(columns="STOP_NUM")

#zoning
GI_data_cut_seefbz = gpd.sjoin(GI_data_cut_seefb,zoning,how="inner", predicate="dwithin",rsuffix='z',distance=20)
aggregated_attributes = (GI_data_cut_seefbz.groupby(GI_data_cut_seefbz.index)["NV5_ZONING"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_seefbz = GI_data_cut_seefb.merge(aggregated_attributes, left_index=True, right_index=True, how="left")

#truck traff
GI_data_cut_seefbzt = gpd.sjoin(GI_data_cut_seefbz,truck_traffic,how="inner", predicate="intersects",rsuffix='f')
aggregated_attributes = (GI_data_cut_seefbzt.groupby(GI_data_cut_seefbzt.index)["TRUCKTRA_4"].agg(lambda x: ", ".join(sorted(set(x.astype(str))))))
GI_data_cut_seefbzt = GI_data_cut_seefbz.merge(aggregated_attributes, left_index=True, right_index=True, how="left")

In [ ]:
#column renaming
GI_data_cut_seefbzt.rename(columns={'TRUCKTRA_4':'Hvy_trucks','NV5_ZONING':'Zoning','STOP_NUM':'Bus_20ft','cat': 'Original_index','TCC_Pct_av':'Tree_cover', 'EC_CST_Ran':'Env_stress', 'Dp2Wtr_ft_':'Gwtr_depth', 'basin_col':'basin_index','LR':'Loading_Rt'}, inplace=True)

In [ ]:
#output
GI_data_out = GI_data_cut_seefbzt.copy()
GI_data_out['Infiltrate'] = None
GI_data_out['BMPs'] = None
GI_data_out['Ranking'] = None

In [ ]:
GI_data_out.to_excel(vector_dir/f'GI_points_{aoi}.xlsx')

In [ ]:
GI_data_out.to_file(vector_dir/f'GI_opportunities_{aoi}.shp')

## END